# RLSF reward-path smoke

---
## 1 — Setup

In [ ]:
# 7B bf16 is ~15 GB of weights and the COMET encoder shares the card: 24 GB is comfortable.
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from pathlib import Path

# %cd into a repo that is already the working directory clones a second copy underneath it,
# so the guard is manage.py, not the directory name.
if not Path('manage.py').exists():
    if not Path('Style-Aware-MT/manage.py').exists():
        !git clone --branch feat/rlsf-implementation https://github.com/prnamhr/Style-Aware-MT.git
    %cd Style-Aware-MT
!git pull --ff-only
!git rev-parse --short HEAD

In [ ]:
import subprocess
import sys

# The one interpreter every shell cell below runs through. `python3` on a rented host is a
# different install from the kernel, and the two stacks drift the moment either is upgraded.
PY = sys.executable
COMET_PY = '.venv-comet/bin/python'
print('kernel', PY)

In [ ]:
# %pip installs into the kernel; !pip may not.
%pip install -q -r requirements.txt

In [ ]:
import numpy
import transformers

# COMET gets its own interpreter. Installing requirements-comet.txt into the kernel downgrades
# transformers and numpy under the generator, which then runs on a stack nothing else uses.
if not Path(COMET_PY).exists():
    pip = [COMET_PY, '-m', 'pip', 'install', '-q']
    subprocess.run([PY, '-m', 'venv', '.venv-comet'], check=True)
    subprocess.run([*pip, '--upgrade', 'pip'], check=True)
    subprocess.run([*pip, 'setuptools<81'], check=True)
    subprocess.run([*pip, '-r', 'requirements-comet.txt'], check=True)
subprocess.run([COMET_PY, '-c', 'import comet; print("comet ok")'], check=True)

print(f'kernel transformers {transformers.__version__}, numpy {numpy.__version__}')
assert transformers.__version__.startswith('5.'), "COMET's pins landed in the kernel"
assert numpy.__version__.startswith('2.'), "COMET's pins landed in the kernel"

In [ ]:
import getpass
import logging
import os

# Set here rather than in a shell cell: the CLI runs and the Kiwi worker are children of this
# kernel and inherit os.environ, so this is the one place the keys have to exist.
for var in ('OPENAI_API_KEY', 'HF_TOKEN'):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f'{var}: ')
logging.getLogger('httpx').setLevel(logging.WARNING)
print({var: bool(os.environ.get(var)) for var in ('OPENAI_API_KEY', 'HF_TOKEN')})

In [ ]:
# wmt22-cometkiwi-da is gated, and the worker is what needs the access; failing here beats
# failing after the generator has already sampled.
check = '''
import os
from huggingface_hub import HfApi, __version__
api, tok = HfApi(), os.environ.get("HF_TOKEN")
print("hub", __version__, "| whoami:", api.whoami(token=tok)["name"])
api.list_repo_files("Unbabel/wmt22-cometkiwi-da", token=tok)
print("cometkiwi access OK")
'''
subprocess.run([COMET_PY, '-c', check], check=True)

---
## 2 — Pre-flight

In [ ]:
import hashlib
import json
import math
from pathlib import Path

import yaml

from src.rlsf.config import judge_concurrency, load_config, reward_config
from src.rlsf.reward import load_train_template
from src.rlsf.smoke import plan

CONFIG   = 'configs/rlsf.yaml'
SEGMENTS = 20

cfg = load_config(CONFIG, require_caps=False)
G   = cfg['rlsf']['rollout']['group_size']

# -- the caps were declared 2026-08-08, so "caps are null" no longer separates this pilot
#    from a training run. What still does is the pilot ceiling: this notebook may spend
#    only pilot.judge_calls, whatever the training caps allow
assert cfg['rlsf']['pilot']['judge_calls'] <= cfg['rlsf']['caps']['max_judge_calls']
assert SEGMENTS * G <= cfg['rlsf']['pilot']['judge_calls'], 'over the pilot ceiling'

# -- the locked control: a quantized or swapped base is a different experiment
gen = cfg['generator']
assert gen['model'] == 'Qwen/Qwen2.5-7B-Instruct', gen['model']
assert gen['load_in_4bit'] is False, 'quantizing redefines the frozen base'
assert gen['adapter_path'], 'RLSF initializes from the frozen PEFT checkpoint'

# -- models/ is not tracked, so a fresh clone has no adapter to initialize from
assert Path(gen['adapter_path'], 'adapter_config.json').exists(), (
    f"{gen['adapter_path']} is missing; copy the frozen PEFT checkpoint onto this host first"
)

# -- greedy rollouts give a group zero variance and the whole run is uninformative
assert cfg['rlsf']['rollout']['temperature'] > 0, 'greedy rollouts cannot be normalized'
assert G <= cfg['rlsf']['caps']['group_size_ceiling']

# -- reward_config rescales to unit ||omega||, so these are not the raw config numbers:
#    unnormalized the weights double as a step size and the grid cells differ by 1.68x
w = reward_config(cfg).weights
print(f"policy {gen['model']} + {gen['adapter_path']}")
print(f"rollout T={cfg['rlsf']['rollout']['temperature']} G={G}")
print('reward', {k: round(v, 3) for k, v in w.items()},
      f"||omega|| = {math.hypot(*w.values()):.3f}")
print(f"judge {cfg['judge']['model']} at concurrency {judge_concurrency(cfg)}")

In [ ]:
# -- the reward judge must not be either evaluation rater, or training spends a rater on
#    the one condition that most needs a rater it was not trained against
raters = {yaml.safe_load(Path(p).read_text())['judge']['model']
          for p in ('configs/judge_eval.yaml', 'configs/judge_eval_gpt.yaml')}
assert cfg['judge']['model'] not in raters, (cfg['judge']['model'], raters)

# -- seeded, because under group normalization a rater flipping a 3 to a 4 inverts an
#    advantage sign
assert cfg['judge']['temperature'] == 0.0 and cfg['judge']['seed'] == 42

# -- the rubric must be the frozen one; load_train_template raises on drift, this reports it
text = load_train_template()
digest = hashlib.sha256(text.encode()).hexdigest()
frozen = json.loads(Path('prompts/hashes.json').read_text())['templates']
assert digest == frozen['judge_train.txt']['sha256']
assert cfg['template_file'] == 'prompts/judge_train.txt', 'the eval rubric would be circular'

print(f"reward judge {cfg['judge']['model']}, distinct from {sorted(raters)}")
print(f"rubric verified {digest[:16]}")

In [ ]:
# -- the dev slice, against the manifest written when it was carved
man = json.loads(Path('data/splits/rlsf_dev_manifest.json').read_text())
for name, want in man['hashes'].items():
    got = hashlib.sha256((Path('data/splits') / name).read_bytes()).hexdigest()
    assert got == want, f'{name} differs from the manifest'
print(f"dev slice {man['counts']['rlsf_dev']} segments, {man['counts']['dev_works']} works")

# -- the slice is not unseen by the model; it selects weights, it does not measure them
print('\n'.join('  ' + c for c in man['caveats']))

p = plan(SEGMENTS, G)
print(f"\nplanned: {p['judge_calls']} judge calls, ~${p['est_usd']} "
      f"(pilot ceiling {cfg['rlsf']['pilot']['judge_calls']})")

---
## 3 — Free pass

Four segments, no judge calls. This is the cell that fails when the Kiwi worker cannot start,
which is the failure the two-interpreter setup exists to avoid.

In [ ]:
!{PY} manage.py rlsf_smoke --config {CONFIG} --segments 4 --skip_judge \
    --out outputs/rlsf/smoke_free.jsonl

---
## 4 — Paid pass

In [ ]:
# Writes outputs/rlsf/smoke_hyps.jsonl alongside the log, so section 6 can re-score without
# sampling again.
!{PY} manage.py rlsf_smoke --config {CONFIG} --segments {SEGMENTS} --group_size {G} --yes

---
## 5 — Read the result

In [ ]:
log = json.loads(Path('outputs/rlsf/smoke_steps.jsonl').read_text().splitlines()[0])
print(f"samples {log['n_samples']}  feasible {log['n_feasible']} "
      f"({log['n_feasible'] / log['n_samples']:.0%})  unmeasured {log['n_unmeasured']}")
print(f"reward mean {log['reward_mean']:+.3f}  sd {log['reward_sd']:.3f}")

# A group whose combined reward is flat contributes no gradient; the verdict printed by the
# run is this fraction against 25%. Absent means a log written before the field existed.
if 'degenerate_frac' in log:
    print(f"degenerate groups {log['degenerate_groups']}/{log['n_groups']} "
          f"({log['degenerate_frac']:.0%}), min group sd {log['min_group_sd']}")
else:
    print('degenerate groups: not recorded in this log, re-run the paid pass')
print(f"length mean {log['length_mean']:.1f} words, ratio to reference "
      f"{log['length_ratio_mean']:.2f}")
print('\nraw component means:', {k: round(v, 3) for k, v in log['raw'].items()})


print('z-deviation from the register centroid:')
for k, v in log['z'].items():
    print(f"  {k:12s} {v:+.2f} +/- {log['z_se'][k]:.2f}")

In [ ]:
# The measured per-call rate, which is what docs/budget.md prices the arm at since 2026-08-08.
u = json.loads(Path('outputs/rlsf/smoke_usage.json').read_text())
print(f"{u['calls']} calls, {u['prompt_tokens'] / u['calls']:.0f} in / "
      f"{u['completion_tokens'] / u['calls']:.0f} out per call")
print(f"${u['per_call_usd']:.6f}/call  (docs/budget.md: $7.375e-05)")
print(f"judge block {u['wall_s']:.1f}s wall at concurrency {u['concurrency']}, "
      f"{u['achieved_parallelism']:.1f}x achieved -- that wall clock is the per-step GPU idle")
print(f"\nplanned 35,992 calls -> ${u['per_call_usd'] * 35_992:.2f}  (budget.md: $2.65)")
print(f"cap     340,000 calls -> ${u['per_call_usd'] * 340_000:.2f}  (cap: $25.00)")

---
## 6 — Re-score without sampling (optional, free)


In [ ]:
!{PY} manage.py rlsf_smoke --config {CONFIG} --segments {SEGMENTS} --group_size {G} \
    --hyps_file outputs/rlsf/smoke_hyps.jsonl --skip_judge \
    --out outputs/rlsf/smoke_rescore.jsonl